In [1]:
import pandas as pd
import numpy as np
import pickle
import wandb
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

In [3]:
# Device setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version : 2.10.0+cu128
Device          : cuda
CUDA available  : True
GPU: Tesla T4


In [5]:
CFG = {
    # Vocabulary
    'vocab_size'    : 15000,   # how many unique words to keep
    'max_len'       : 256,     # max tokens per sample

    # Model architecture
    'embed_dim'     : 128,     # word embedding dimensions
    'hidden_dim'    : 256,     # LSTM hidden state size
    'num_layers'    : 2,       # number of LSTM layers
    'dropout'       : 0.3,     # dropout between LSTM layers
    'num_classes'   : 5,       # A, B, C, D, E

    # Training
    'batch_size'    : 32,
    'learning_rate' : 1e-3,
    'epochs'        : 50,
    'patience'      : 8,       # early stopping patience
    'weight_decay'  : 1e-4,    # L2 regularization
    'random_state'  : 42,

    # Paths
    'data_dir'   : '/kaggle/input/competitions/smart-mcq-solver-challenge',
    'output_dir' : '/kaggle/working/outputs',
}

In [6]:
torch.manual_seed(CFG['random_state'])
np.random.seed(CFG['random_state'])

OUTPUT_DIR = Path(CFG['output_dir'])
DATA_DIR   = Path(CFG['data_dir'])

print('Config set!')
for k, v in CFG.items():
    print(f'   {k:<16}: {v}')

Config set!
   vocab_size      : 15000
   max_len         : 256
   embed_dim       : 128
   hidden_dim      : 256
   num_layers      : 2
   dropout         : 0.3
   num_classes     : 5
   batch_size      : 32
   learning_rate   : 0.001
   epochs          : 50
   patience        : 8
   weight_decay    : 0.0001
   random_state    : 42
   data_dir        : /kaggle/input/competitions/smart-mcq-solver-challenge
   output_dir      : /kaggle/working/outputs


In [7]:
raw_train = pd.read_csv(DATA_DIR / 'train.csv')
raw_test  = pd.read_csv(DATA_DIR / 'test.csv')

In [8]:
# Lowercase all the text
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_cols:
    raw_train[col] = raw_train[col].str.lower().str.strip()
    raw_test[col]  = raw_test[col].str.lower().str.strip()

In [9]:
# Stratified 80/20 split
np.random.seed(CFG['random_state'])
train_idx, val_idx = [], []
for ans in 'ABCDE':
    idx = raw_train[raw_train['answer'] == ans].index.tolist()
    np.random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_idx += idx[:cut]
    val_idx   += idx[cut:]

train_df = raw_train.loc[train_idx].reset_index(drop=True)
val_df   = raw_train.loc[val_idx].reset_index(drop=True)
test_df  = raw_test.copy()

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 1599 | Val: 401 | Test: 500


In [10]:
# Answer label maps
ANSWER_MAP  = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
REVERSE_MAP = {v: k for k, v in ANSWER_MAP.items()}

Simple word-level tokenizer built from scratch.

    Workflow:
      1. build_vocab()  — scan all training texts, keep top-N words
      2. encode()       — text → list of integer token IDs
      3. pad_or_truncate() — make every sequence the same length

    Special tokens:
      <PAD> = 0  (padding for shorter sequences)
      <UNK> = 1  (unknown words not in vocab)

In [11]:
class MCQTokenizer:

    PAD_TOKEN = '<PAD>'
    UNK_TOKEN = '<UNK>'

    def __init__(self, vocab_size: int = 10000):
        self.vocab_size = vocab_size
        self.word2idx   = {self.PAD_TOKEN: 0, self.UNK_TOKEN: 1}
        self.idx2word   = {0: self.PAD_TOKEN, 1: self.UNK_TOKEN}
        self.vocab_built = False

    def tokenize(self, text: str) -> List[str]:
        """Split text into lowercase words (simple word-level tokenization)"""
        return str(text).lower().split()

    def build_vocab(self, texts: List[str]):
        """
        Build vocabulary from a list of training texts.
        Keeps only top vocab_size most frequent words.
        """
        counter = Counter()
        for text in texts:
            counter.update(self.tokenize(text))

        # Keep top (vocab_size - 2) words (reserve 0, 1 for PAD, UNK)
        most_common = counter.most_common(self.vocab_size - 2)
        for idx, (word, _) in enumerate(most_common, start=2):
            self.word2idx[word] = idx
            self.idx2word[idx]  = word

        self.vocab_built = True
        print(f'  Vocab built: {len(self.word2idx)} tokens')

    def encode(self, text: str) -> List[int]:
        """Convert text to list of integer IDs"""
        words = self.tokenize(text)
        return [self.word2idx.get(w, 1) for w in words]  # 1 = <UNK>

    def pad_or_truncate(self, ids: List[int], max_len: int) -> List[int]:
        """Make sequence exactly max_len long (truncate or pad with 0)"""
        if len(ids) >= max_len:
            return ids[:max_len]
        return ids + [0] * (max_len - len(ids))  # 0 = <PAD>


# Build tokenizer on training data
print('Building tokenizer...')
tokenizer = MCQTokenizer(vocab_size=CFG['vocab_size'])

# Collect all training texts (prompt + options)
all_train_texts = []
for _, row in train_df.iterrows():
    all_train_texts.append(
        f"{row['prompt']} {row['A']} {row['B']} {row['C']} {row['D']} {row['E']}"
    )

tokenizer.build_vocab(all_train_texts)
print(f'Tokenizer ready! Vocab size = {len(tokenizer.word2idx)}')

Building tokenizer...
  Vocab built: 3815 tokens
Tokenizer ready! Vocab size = 3815


### PYTORCH DATASET

    Custom PyTorch Dataset for MCQ questions.

    Each sample:
      - combines prompt + all 5 options into one long text
      - tokenizes it with our custom tokenizer
      - pads/truncates to max_len
      - returns (input_ids tensor, label tensor)

In [12]:
class MCQDataset(Dataset):
    def __init__(self ,df ,tokenizer ,max_len ,is_test):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Build combined text: prompt + all options
        text = (f"{row['prompt']} "
                f"{row['A']} {row['B']} {row['C']} {row['D']} {row['E']}")

        # Tokenize → encode → pad/truncate
        ids     = self.tokenizer.encode(text)
        ids     = self.tokenizer.pad_or_truncate(ids, self.max_len)
        id_tensor = torch.tensor(ids, dtype=torch.long)

        if self.is_test:
            return id_tensor   # no label for test

        # Convert answer letter to int label
        label = ANSWER_MAP[row['answer']]
        return id_tensor, torch.tensor(label, dtype=torch.long)

In [13]:
# Create datasets
train_dataset = MCQDataset(train_df, tokenizer, CFG['max_len'], is_test=False)
val_dataset   = MCQDataset(val_df,   tokenizer, CFG['max_len'], is_test=False)
test_dataset  = MCQDataset(test_df,  tokenizer, CFG['max_len'], is_test=True)

In [14]:
# Create DataLoaders 
train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=0, pin_memory=True)

In [15]:
print(f'Datasets and DataLoaders ready!')
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

Datasets and DataLoaders ready!
Train batches : 50
Val batches   : 13
Test batches  : 16


In [16]:
# Quick sanity check on one batch
sample_ids, sample_labels = next(iter(train_loader))
print(f'Sample batch  : ids={sample_ids.shape}, labels={sample_labels.shape}')

Sample batch  : ids=torch.Size([32, 256]), labels=torch.Size([32])


### LSTM MODEL (FROM SCRATCH)

    2-layer Bidirectional LSTM for MCQ answer ranking.

    Architecture:
      [Input IDs]  →  Embedding  →  LSTM x2  →  Dropout  →  Linear  →  5 logits

    Why Bidirectional?
      - Forward LSTM reads left→right (question context)
      - Backward LSTM reads right→left (option context)
      - Concatenating both gives best results

In [17]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size , embed_dim , hidden_dim , num_layers , num_classes , dropout , pad_idx:int = 0 ):
        super().__init__()
        
        # Layer 1: Embedding
        # Maps each token ID → dense embed_dim-dimensional vector
        # padding_idx=0 ensures <PAD> tokens don't affect gradients
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim  = embed_dim,
            padding_idx    = pad_idx
        )

        # Layer 2: LSTM
        # bidirectional=True → output is hidden_dim * 2
        # batch_first=True   → input shape: (batch, seq_len, embed_dim)
        self.lstm = nn.LSTM(
            input_size    = embed_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0
        )
        
        # Layer 3: Dropout for regularization
        self.dropout = nn.Dropout(dropout)

        # Layer 4: Fully-connected head
        # hidden_dim*2 because bidirectional → concat forward + backward
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:

        # Step 1: Embed token IDs → word vectors
        embedded = self.embedding(input_ids)
        embedded = self.dropout(embedded)

        # Step 2: Pass through LSTM
        output, (hidden, cell) = self.lstm(embedded)

        # Step 3: Pool — take last hidden states from both directions
        forward_hidden  = hidden[-2]   # last layer, forward
        backward_hidden = hidden[-1]   # last layer, backward
        combined = torch.cat([forward_hidden, backward_hidden], dim=1)

        # Step 4: Dropout + linear projection → class logits
        out = self.dropout(combined)
        logits = self.fc(out)          # (batch, 5)

        return logits

In [18]:
# Instantiate model
model = LSTMClassifier(
    vocab_size  = CFG['vocab_size'],
    embed_dim   = CFG['embed_dim'],
    hidden_dim  = CFG['hidden_dim'],
    num_layers  = CFG['num_layers'],
    num_classes = CFG['num_classes'],
    dropout     = CFG['dropout'],
).to(DEVICE)

In [19]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f' LSTM model created!')
print(f'  Architecture: Embedding → BiLSTM x2 → Dropout → FC')
print(f'  Total params: {total_params:,}')
print(f'  Device      : {DEVICE}')
print(model)

 LSTM model created!
  Architecture: Embedding → BiLSTM x2 → Dropout → FC
  Total params: 4,290,053
  Device      : cuda
LSTMClassifier(
  (embedding): Embedding(15000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=5, bias=True)
)


### TRAINING UTILITIES

In [20]:
### 1.Calculate MAP@3 directly from raw logits and label tensors

def map_at_3_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    probs  = torch.softmax(logits, dim=1).cpu().numpy()
    labels = labels.cpu().numpy()
    scores = []
    for i, true in enumerate(labels):
        top3 = np.argsort(probs[i])[-3:][::-1]
        score = (1.0 / (np.where(top3 == true)[0][0] + 1)
                 if true in top3 else 0.0)
        scores.append(score)
    return float(np.mean(scores))

In [21]:
### 2. Train for one epoch, return avg loss and MAP@3

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, all_logits, all_labels = 0.0, [], []

    for ids, labels in loader:
        ids, labels = ids.to(device), labels.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass (faster on GPU)
        with autocast(enabled=(device.type == 'cuda')):
            logits = model(ids)
            loss   = criterion(logits, labels)

        # Backward with gradient scaling
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss  += loss.item()
        all_logits.append(logits.detach())
        all_labels.append(labels.detach())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    avg_loss   = total_loss / len(loader)
    train_map3 = map_at_3_from_logits(all_logits, all_labels)

    return avg_loss, train_map3

In [22]:
### 3. Evaluate on val/test, return loss and MAP@3

@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []

    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            ids, labels = batch
            ids, labels = ids.to(device), labels.to(device)
            logits = model(ids)
            total_loss += criterion(logits, labels).item()
            all_labels.append(labels)
        else:
            ids = batch.to(device)
            logits = model(ids)

        all_logits.append(logits)

    all_logits = torch.cat(all_logits)
    avg_loss   = total_loss / len(loader)

    if all_labels:
        all_labels = torch.cat(all_labels)
        val_map3   = map_at_3_from_logits(all_logits, all_labels)
    else:
        val_map3 = 0.0

    return avg_loss, val_map3, all_logits

print('Training utilities done !')

Training utilities done !
